In [1]:
import pandas as pd
import numpy as np
import plotly
import plotly.io as pio
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

# Fix Plotly rendering in Kaggle
pio.renderers.default = "iframe"

print("All required libraries imported successfully!")
print("Plotly version:", plotly.__version__)
print("Plotly renderer:", pio.renderers.default)

All required libraries imported successfully!
Plotly version: 5.24.1
Plotly renderer: iframe


In [2]:
# Load the World Happiness Report 2023 dataset
DATA_PATH = "/kaggle/input/datasets/ajaypalsinghlo/world-happiness-report-2023/WHR2023.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Original dataset shape:", df.shape)

# Rename columns
column_mapping = {
    "Country name": "Country",
    "Ladder score": "Happiness_Score",
    "Logged GDP per capita": "GDP",
    "Social support": "Social_Support",
    "Healthy life expectancy": "Life_Expectancy",
    "Freedom to make life choices": "Freedom",
    "Generosity": "Generosity",
    "Perceptions of corruption": "Corruption"
}

df.rename(columns=column_mapping, inplace=True)

# Check required columns
required_columns = [
    "Country",
    "Happiness_Score",
    "GDP",
    "Freedom"
]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# Convert numerical columns
numeric_columns = [
    "Happiness_Score",
    "GDP",
    "Social_Support",
    "Life_Expectancy",
    "Freedom",
    "Generosity",
    "Corruption"
]

for column in numeric_columns:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

# Clean country names
df["Country"] = df["Country"].astype("string").str.strip()

# Normalize common country-name variations
country_aliases = {
    "Türkiye": "Turkey",
    "Turkiye": "Turkey",
    "State of Palestine": "Palestinian Territories",
    "Congo (Brazzaville)": "Congo",
    "Congo (Kinshasa)": "Democratic Republic of the Congo"
}

df["Country"] = df["Country"].replace(country_aliases)

# Remove rows without country names or happiness scores
df = df.dropna(subset=["Country", "Happiness_Score"]).copy()

# Remove duplicate country entries
df = df.drop_duplicates(subset=["Country"], keep="first").copy()

# Assign countries to geographical regions
region_mapping = {
    # Western Europe
    "Austria": "Western Europe",
    "Belgium": "Western Europe",
    "Denmark": "Western Europe",
    "Finland": "Western Europe",
    "France": "Western Europe",
    "Germany": "Western Europe",
    "Iceland": "Western Europe",
    "Ireland": "Western Europe",
    "Luxembourg": "Western Europe",
    "Netherlands": "Western Europe",
    "Norway": "Western Europe",
    "Portugal": "Western Europe",
    "Spain": "Western Europe",
    "Sweden": "Western Europe",
    "Switzerland": "Western Europe",
    "United Kingdom": "Western Europe",
    "Italy": "Western Europe",
    "Greece": "Western Europe",
    "Cyprus": "Western Europe",
    "Malta": "Western Europe",

    # Latin America
    "Argentina": "Latin America",
    "Bolivia": "Latin America",
    "Brazil": "Latin America",
    "Chile": "Latin America",
    "Colombia": "Latin America",
    "Costa Rica": "Latin America",
    "Dominican Republic": "Latin America",
    "Ecuador": "Latin America",
    "El Salvador": "Latin America",
    "Guatemala": "Latin America",
    "Honduras": "Latin America",
    "Jamaica": "Latin America",
    "Mexico": "Latin America",
    "Nicaragua": "Latin America",
    "Panama": "Latin America",
    "Paraguay": "Latin America",
    "Peru": "Latin America",
    "Uruguay": "Latin America",
    "Venezuela": "Latin America",

    # East Asia
    "China": "East Asia",
    "Japan": "East Asia",
    "Mongolia": "East Asia",
    "South Korea": "East Asia",
    "Taiwan Province of China": "East Asia",
    "Hong Kong S.A.R. of China": "East Asia",

    # Sub-Saharan Africa
    "Benin": "Sub-Saharan Africa",
    "Botswana": "Sub-Saharan Africa",
    "Burkina Faso": "Sub-Saharan Africa",
    "Burundi": "Sub-Saharan Africa",
    "Cameroon": "Sub-Saharan Africa",
    "Chad": "Sub-Saharan Africa",
    "Comoros": "Sub-Saharan Africa",
    "Congo": "Sub-Saharan Africa",
    "Democratic Republic of the Congo": "Sub-Saharan Africa",
    "Ethiopia": "Sub-Saharan Africa",
    "Gabon": "Sub-Saharan Africa",
    "Gambia": "Sub-Saharan Africa",
    "Ghana": "Sub-Saharan Africa",
    "Guinea": "Sub-Saharan Africa",
    "Ivory Coast": "Sub-Saharan Africa",
    "Kenya": "Sub-Saharan Africa",
    "Lesotho": "Sub-Saharan Africa",
    "Liberia": "Sub-Saharan Africa",
    "Madagascar": "Sub-Saharan Africa",
    "Malawi": "Sub-Saharan Africa",
    "Mali": "Sub-Saharan Africa",
    "Mauritania": "Sub-Saharan Africa",
    "Mauritius": "Sub-Saharan Africa",
    "Mozambique": "Sub-Saharan Africa",
    "Namibia": "Sub-Saharan Africa",
    "Niger": "Sub-Saharan Africa",
    "Nigeria": "Sub-Saharan Africa",
    "Senegal": "Sub-Saharan Africa",
    "Sierra Leone": "Sub-Saharan Africa",
    "South Africa": "Sub-Saharan Africa",
    "Tanzania": "Sub-Saharan Africa",
    "Togo": "Sub-Saharan Africa",
    "Uganda": "Sub-Saharan Africa",
    "Zambia": "Sub-Saharan Africa",
    "Zimbabwe": "Sub-Saharan Africa",

    # South Asia
    "Afghanistan": "South Asia",
    "Bangladesh": "South Asia",
    "India": "South Asia",
    "Nepal": "South Asia",
    "Pakistan": "South Asia",
    "Sri Lanka": "South Asia",

    # Southeast Asia
    "Cambodia": "Southeast Asia",
    "Indonesia": "Southeast Asia",
    "Laos": "Southeast Asia",
    "Malaysia": "Southeast Asia",
    "Myanmar": "Southeast Asia",
    "Philippines": "Southeast Asia",
    "Singapore": "Southeast Asia",
    "Thailand": "Southeast Asia",
    "Vietnam": "Southeast Asia",

    # Central and Eastern Europe
    "Albania": "Central and Eastern Europe",
    "Armenia": "Central and Eastern Europe",
    "Azerbaijan": "Central and Eastern Europe",
    "Belarus": "Central and Eastern Europe",
    "Bosnia and Herzegovina": "Central and Eastern Europe",
    "Bulgaria": "Central and Eastern Europe",
    "Croatia": "Central and Eastern Europe",
    "Czechia": "Central and Eastern Europe",
    "Estonia": "Central and Eastern Europe",
    "Georgia": "Central and Eastern Europe",
    "Hungary": "Central and Eastern Europe",
    "Kazakhstan": "Central and Eastern Europe",
    "Kosovo": "Central and Eastern Europe",
    "Kyrgyzstan": "Central and Eastern Europe",
    "Latvia": "Central and Eastern Europe",
    "Lithuania": "Central and Eastern Europe",
    "Moldova": "Central and Eastern Europe",
    "Montenegro": "Central and Eastern Europe",
    "North Macedonia": "Central and Eastern Europe",
    "Poland": "Central and Eastern Europe",
    "Romania": "Central and Eastern Europe",
    "Russia": "Central and Eastern Europe",
    "Serbia": "Central and Eastern Europe",
    "Slovakia": "Central and Eastern Europe",
    "Slovenia": "Central and Eastern Europe",
    "Tajikistan": "Central and Eastern Europe",
    "Ukraine": "Central and Eastern Europe",
    "Uzbekistan": "Central and Eastern Europe",

    # Middle East and North Africa
    "Algeria": "Middle East and North Africa",
    "Bahrain": "Middle East and North Africa",
    "Egypt": "Middle East and North Africa",
    "Iran": "Middle East and North Africa",
    "Iraq": "Middle East and North Africa",
    "Israel": "Middle East and North Africa",
    "Jordan": "Middle East and North Africa",
    "Kuwait": "Middle East and North Africa",
    "Lebanon": "Middle East and North Africa",
    "Libya": "Middle East and North Africa",
    "Morocco": "Middle East and North Africa",
    "Oman": "Middle East and North Africa",
    "Palestinian Territories": "Middle East and North Africa",
    "Qatar": "Middle East and North Africa",
    "Saudi Arabia": "Middle East and North Africa",
    "Syria": "Middle East and North Africa",
    "Tunisia": "Middle East and North Africa",
    "Turkey": "Middle East and North Africa",
    "United Arab Emirates": "Middle East and North Africa",
    "Yemen": "Middle East and North Africa",

    # North America, Australia and New Zealand
    "Australia": "North America and ANZ",
    "Canada": "North America and ANZ",
    "New Zealand": "North America and ANZ",
    "United States": "North America and ANZ"
}

# Apply region classification
df["Region"] = df["Country"].map(region_mapping)

# Identify unmapped countries
unmapped_countries = sorted(
    df.loc[df["Region"].isna(), "Country"].unique()
)

if unmapped_countries:
    print("Countries without a region mapping:", unmapped_countries)
else:
    print("All countries have been assigned a region.")

# Do not silently discard countries if mapping is incomplete
if unmapped_countries:
    raise ValueError(
        "Please add the unmapped countries to region_mapping before continuing."
    )

print("\nData preparation completed.")
print("Final dataset shape:", df.shape)
print("Number of countries:", df["Country"].nunique())
print("Number of regions:", df["Region"].nunique())

print("\nCountries per region:")
print(df["Region"].value_counts())

print("\nMissing values in key columns:")
print(
    df[["Country", "Region", "Happiness_Score", "GDP", "Freedom"]]
    .isna()
    .sum()
)

display(df.head())

Dataset loaded successfully!
Original dataset shape: (137, 19)
All countries have been assigned a region.

Data preparation completed.
Final dataset shape: (137, 20)
Number of countries: 137
Number of regions: 9

Countries per region:
Region
Sub-Saharan Africa              33
Central and Eastern Europe      26
Western Europe                  20
Latin America                   19
Middle East and North Africa    14
Southeast Asia                   9
East Asia                        6
South Asia                       6
North America and ANZ            4
Name: count, dtype: int64

Missing values in key columns:
Country            0
Region             0
Happiness_Score    0
GDP                0
Freedom            0
dtype: int64


,Country,Happiness_Score,Standard error of ladder score,upperwhisker,lowerwhisker,GDP,Social_Support,Life_Expectancy,Freedom,Generosity,Corruption,Ladder score in Dystopia,Explained by: Log GDP per capita,Explained by: Social support,Explained by: Healthy life expectancy,Explained by: Freedom to make life choices,Explained by: Generosity,Explained by: Perceptions of corruption,Dystopia + residual,Region
0,Finland,7.804,0.036,7.875,7.733,10.792,0.969,71.150,0.961,-0.019,0.182,1.778,1.888,1.585,0.535,0.772,0.126,0.535,2.363,Western Europe
1,Denmark,7.586,0.041,7.667,7.506,10.962,0.954,71.250,0.934,0.134,0.196,1.778,1.949,1.548,0.537,0.734,0.208,0.525,2.084,Western Europe
2,Iceland,7.530,0.049,7.625,7.434,10.896,0.983,72.050,0.936,0.211,0.668,1.778,1.926,1.620,0.559,0.738,0.250,0.187,2.250,Western Europe
3,Israel,7.473,0.032,7.535,7.411,10.639,0.943,72.697,0.809,-0.023,0.708,1.778,1.833,1.521,0.577,0.569,0.124,0.158,2.691,Middle East and North Africa
4,Netherlands,7.403,0.029,7.460,7.346,10.942,0.930,71.550,0.887,0.213,0.379,1.778,1.942,1.488,0.545,0.672,0.251,0.394,2.110,Western Europe


In [3]:
# Task 1: Average Happiness Score by Region

# Calculate average happiness score by region
regional_happiness = (
    df.groupby("Region", as_index=False)["Happiness_Score"]
    .mean()
    .sort_values("Happiness_Score", ascending=True)
)

# Identify the region with the highest average happiness score
top_region_row = regional_happiness.loc[
    regional_happiness["Happiness_Score"].idxmax()
]

top_region = top_region_row["Region"]
top_score = top_region_row["Happiness_Score"]

# Assign colors
regional_happiness["Color"] = regional_happiness["Region"].apply(
    lambda region: "#2878B5"
    if region == top_region
    else "#A7C7E7"
)

# Create the horizontal bar chart
fig1 = go.Figure(
    go.Bar(
        x=regional_happiness["Happiness_Score"],
        y=regional_happiness["Region"],
        orientation="h",
        marker_color=regional_happiness["Color"],
        text=regional_happiness["Happiness_Score"].round(2),
        textposition="outside",
        hovertemplate=(
            "Region: %{y}<br>"
            "Average happiness score: %{x:.2f}"
            "<extra></extra>"
        )
    )
)

fig1.update_layout(
    title=f"Average Happiness Score by Region — Highest: {top_region}",
    xaxis_title="Average Happiness Score",
    yaxis_title="Region",
    template="plotly_white",
    height=600,
    margin=dict(l=30, r=80, t=90, b=50),
    showlegend=False
)

fig1.update_xaxes(
    rangemode="tozero",
    showline=True,
    zeroline=True,
    zerolinewidth=2,
    zerolinecolor="black"
)

# Display using the working iframe renderer
fig1.show(renderer="iframe")

print(f"Highest-scoring region: {top_region} ({top_score:.2f})")

print("\nRegional average happiness scores:")
display(
    regional_happiness[["Region", "Happiness_Score"]]
    .sort_values("Happiness_Score", ascending=False)
    .round(2)
)

Highest-scoring region: North America and ANZ (7.02)

Regional average happiness scores:


,Region,Happiness_Score
4,North America and ANZ,7.02
8,Western Europe,6.89
2,Latin America,5.97
0,Central and Eastern Europe,5.95
1,East Asia,5.93
6,Southeast Asia,5.43
3,Middle East and North Africa,5.10
7,Sub-Saharan Africa,4.38
5,South Asia,4.09


In [4]:
# Task 2: Top 8 vs Bottom 8 Happiest Countries

# Calculate the global average
global_average = df["Happiness_Score"].mean()

# Select the top and bottom eight countries
top_8 = (
    df.nlargest(8, "Happiness_Score")
    [["Country", "Happiness_Score"]]
    .copy()
)

bottom_8 = (
    df.nsmallest(8, "Happiness_Score")
    [["Country", "Happiness_Score"]]
    .copy()
)

top_8["Group"] = "Top 8 Happiest"
bottom_8["Group"] = "Bottom 8 Happiest"

# Combine the two groups
comparison = pd.concat(
    [top_8, bottom_8],
    ignore_index=True
)

comparison = comparison.sort_values(
    "Happiness_Score",
    ascending=True
)

# Set colors
group_colors = {
    "Top 8 Happiest": "#2878B5",
    "Bottom 8 Happiest": "#E87532"
}

comparison["Color"] = comparison["Group"].map(group_colors)

# Calculate the happiness gap
highest_score = df["Happiness_Score"].max()
lowest_score = df["Happiness_Score"].min()
happiness_gap = highest_score - lowest_score

# Create the chart
fig2 = go.Figure(
    go.Bar(
        x=comparison["Happiness_Score"],
        y=comparison["Country"],
        orientation="h",
        marker_color=comparison["Color"],
        text=comparison["Happiness_Score"].round(2),
        textposition="outside",
        customdata=comparison[["Group"]],
        hovertemplate=(
            "Country: %{y}<br>"
            "Happiness score: %{x:.2f}<br>"
            "Group: %{customdata[0]}"
            "<extra></extra>"
        )
    )
)

# Add global average reference line
fig2.add_vline(
    x=global_average,
    line_dash="dash",
    line_color="black",
    line_width=2,
    annotation_text=f"Global average: {global_average:.2f}",
    annotation_position="top"
)

# Add happiness-gap annotation
fig2.add_annotation(
    x=0.5,
    y=1.12,
    xref="paper",
    yref="paper",
    text=f"Highest-to-lowest happiness gap: {happiness_gap:.2f} points",
    showarrow=False
)

fig2.update_layout(
    title="Top 8 vs Bottom 8 Happiest Countries — 2023",
    xaxis_title="Happiness Score",
    yaxis_title="Country",
    template="plotly_white",
    height=750,
    margin=dict(l=30, r=60, t=120, b=50),
    showlegend=False
)

fig2.update_xaxes(
    rangemode="tozero",
    showline=True,
    zeroline=True,
    zerolinewidth=2,
    zerolinecolor="black"
)

# Display using the working iframe renderer
fig2.show(renderer="iframe")

print(f"Global average happiness score: {global_average:.2f}")
print(f"Highest score: {highest_score:.2f}")
print(f"Lowest score: {lowest_score:.2f}")
print(f"Happiness gap: {happiness_gap:.2f}")

Global average happiness score: 5.54
Highest score: 7.80
Lowest score: 1.86
Happiness gap: 5.95


In [5]:
# Task 3: GDP vs Freedom Across Selected Regions

# Select the regions
selected_regions = [
    "Western Europe",
    "Latin America",
    "East Asia",
    "Sub-Saharan Africa",
    "South Asia"
]

# Filter the dataset
selected_df = df[
    df["Region"].isin(selected_regions)
].copy()

# Calculate regional averages
regional_comparison = (
    selected_df.groupby("Region", as_index=False)
    .agg(
        Average_GDP=("GDP", "mean"),
        Average_Freedom=("Freedom", "mean")
    )
)

# Preserve the specified region order
regional_comparison["Region"] = pd.Categorical(
    regional_comparison["Region"],
    categories=selected_regions,
    ordered=True
)

regional_comparison = regional_comparison.sort_values("Region")

# Create the grouped bar chart
fig3 = go.Figure()

fig3.add_trace(
    go.Bar(
        x=regional_comparison["Region"],
        y=regional_comparison["Average_GDP"],
        name="Average GDP per Capita",
        marker_color="#2878B5",
        hovertemplate=(
            "Region: %{x}<br>"
            "Average GDP per capita: %{y:.2f}"
            "<extra></extra>"
        )
    )
)

fig3.add_trace(
    go.Bar(
        x=regional_comparison["Region"],
        y=regional_comparison["Average_Freedom"],
        name="Average Freedom Score",
        marker_color="#E87532",
        hovertemplate=(
            "Region: %{x}<br>"
            "Average freedom score: %{y:.3f}"
            "<extra></extra>"
        )
    )
)

fig3.update_layout(
    title="GDP vs Freedom Across Selected Regions",
    xaxis_title="Region",
    yaxis_title="Average Value",
    barmode="group",
    template="plotly_white",
    height=600,
    margin=dict(l=40, r=30, t=80, b=120),
    legend_title_text="Metric"
)

fig3.update_xaxes(tickangle=-25)

fig3.update_yaxes(
    rangemode="tozero",
    showline=True,
    zeroline=True,
    zerolinewidth=2,
    zerolinecolor="black"
)

# Display using the working iframe renderer
fig3.show(renderer="iframe")

display(regional_comparison)

,Region,Average_GDP,Average_Freedom
4,Western Europe,10.855200,0.853150
1,Latin America,9.290053,0.836895
0,East Asia,10.363667,0.765000
3,Sub-Saharan Africa,8.071879,0.703485
2,South Asia,8.509167,0.747500
